# 03 — Preprocesamiento

**Objetivo del notebook:** transformar los datos crudos en datos listos para modelar, tomando y documentando todas las decisiones de preparación.

Pasos:
1. Eliminar columnas inútiles (`ID`, `oral`).
2. Definir **X** (features) e **y** (target).
3. Dividir en train (80%) y test (20%) de forma **estratificada**.
4. Codificar variables categóricas (`gender`, `tartar`) con OneHotEncoder.
5. Escalar features (versión escalada para KNN).
6. Guardar todo lo procesado y los transformadores en disco.

**Principio rector:** todos los transformadores (`OneHotEncoder`, `MinMaxScaler`) se ajustan (`fit`) **solo con los datos de entrenamiento** y luego se aplican (`transform`) tanto a train como a test. Esto evita el *data leakage* — que información del conjunto de test se filtre al entrenamiento.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

df = pd.read_excel('../data/raw/smoking_prediction.xlsx')
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

Dataset cargado: 50,000 filas × 27 columnas


## 1. Eliminación de columnas inútiles

Del EDA concluimos que `ID` (identificador) y `oral` (un solo valor) no aportan señal. Las eliminamos.

In [2]:
columnas_a_eliminar = ['ID', 'oral']
df = df.drop(columns=columnas_a_eliminar)

print(f'Columnas eliminadas: {columnas_a_eliminar}')
print(f'Columnas restantes: {df.shape[1]} ({df.shape[1]-1} features + 1 target)')

Columnas eliminadas: ['ID', 'oral']
Columnas restantes: 25 (24 features + 1 target)


## 2. Definición de X e y

**y** (lo que queremos predecir) es la columna `smoking`.

**X** (lo que usamos para predecir) son todas las demás columnas. Esta es una decisión clave: usamos todas las features disponibles (menos las eliminadas) y dejamos que los modelos determinen cuáles pesan más, en lugar de descartar variables manualmente. Los modelos de árboles hacen selección de features de forma natural.

In [3]:
X = df.drop(columns=['smoking'])
y = df['smoking']

print(f'X (features): {X.shape[1]} columnas')
print(f'y (target):   {y.name}')
print()
print('Features:')
print(list(X.columns))

X (features): 24 columnas
y (target):   smoking

Features:
['gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries', 'tartar']


## 3. Identificación de tipos de columna

Separamos las columnas categóricas (que necesitan encoding) de las numéricas (que no). Esto define cómo tratamos cada grupo.

In [4]:
cat_cols = ['gender', 'tartar']
num_cols = [c for c in X.columns if c not in cat_cols]

print(f'Columnas categóricas ({len(cat_cols)}): {cat_cols}')
print(f'Columnas numéricas ({len(num_cols)}): {num_cols}')

Columnas categóricas (2): ['gender', 'tartar']
Columnas numéricas (22): ['age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries']


## 4. Split train / test estratificado (80/20)

Dividimos antes de codificar y escalar, para evitar data leakage. Usamos:
- `test_size=0.2` → 80% entrenamiento, 20% test (como pide la consigna).
- `stratify=y` → mantiene la proporción de fumadores/no fumadores en ambos conjuntos. Crítico en datasets desbalanceados.
- `random_state=42` → reproducibilidad: el split es siempre el mismo.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train: {X_train.shape[0]:,} filas')
print(f'X_test:  {X_test.shape[0]:,} filas')
print()
print('Proporción del target en cada conjunto:')
print(f'  Train → No fuma: {(y_train==0).mean()*100:.1f}%  |  Fuma: {(y_train==1).mean()*100:.1f}%')
print(f'  Test  → No fuma: {(y_test==0).mean()*100:.1f}%  |  Fuma: {(y_test==1).mean()*100:.1f}%')
print()
print('→ Las proporciones son idénticas gracias a stratify.')

X_train: 40,000 filas
X_test:  10,000 filas

Proporción del target en cada conjunto:
  Train → No fuma: 63.3%  |  Fuma: 36.7%
  Test  → No fuma: 63.3%  |  Fuma: 36.7%

→ Las proporciones son idénticas gracias a stratify.


## 5. Codificación de categóricas (OneHotEncoder)

`OneHotEncoder` convierte cada categoría en columnas binarias. Lo configuramos con:
- `handle_unknown='ignore'` → si en los datos de entrega aparece una categoría no vista en train, no rompe — la codifica como todo ceros.
- `sparse_output=False` → devuelve un array denso (normal de numpy) en lugar de una matriz dispersa, más fácil de manejar.

**Lo ajustamos solo con X_train** y lo aplicamos a ambos conjuntos.

In [6]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# fit SOLO con train
ohe.fit(X_train[cat_cols])

# transform en ambos
X_train_ohe = ohe.transform(X_train[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

nombres_ohe = ohe.get_feature_names_out(cat_cols)
print(f'Categóricas codificadas en {len(nombres_ohe)} columnas:')
print(list(nombres_ohe))

Categóricas codificadas en 4 columnas:
['gender_F', 'gender_M', 'tartar_N', 'tartar_Y']


In [7]:
# Reconstruimos los DataFrames combinando numéricas + categóricas codificadas
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=nombres_ohe, index=X_train.index)
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=nombres_ohe, index=X_test.index)

X_train_final = pd.concat([X_train[num_cols], X_train_ohe_df], axis=1)
X_test_final = pd.concat([X_test[num_cols], X_test_ohe_df], axis=1)

print(f'X_train_final: {X_train_final.shape}')
print(f'X_test_final:  {X_test_final.shape}')
print()
print('Columnas finales:')
print(list(X_train_final.columns))

X_train_final: (40000, 26)
X_test_final:  (10000, 26)

Columnas finales:
['age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries', 'gender_F', 'gender_M', 'tartar_N', 'tartar_Y']


## 6. Versión escalada (para KNN)

KNN se basa en distancias entre puntos, por lo que es muy sensible a la escala de las variables. Una variable con valores grandes dominaría el cálculo de distancia. Por eso creamos una versión escalada con `MinMaxScaler`, que lleva todas las variables al rango [0, 1].

Los modelos de árboles (DecisionTree, RandomForest, XGBoost) **no** necesitan escalado, así que para ellos usaremos la versión sin escalar.

De nuevo: el scaler se ajusta solo con train.

In [8]:
scaler = MinMaxScaler()

# fit SOLO con train
scaler.fit(X_train_final)

X_train_scaled = pd.DataFrame(
    scaler.transform(X_train_final),
    columns=X_train_final.columns,
    index=X_train_final.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_final),
    columns=X_test_final.columns,
    index=X_test_final.index
)

print('Versión escalada creada (rango 0-1):')
print(X_train_scaled.describe().T[['min', 'max']].round(2).head(8))

Versión escalada creada (rango 0-1):
                 min  max
age              0.0  1.0
height(cm)       0.0  1.0
weight(kg)       0.0  1.0
waist(cm)        0.0  1.0
eyesight(left)   0.0  1.0
eyesight(right)  0.0  1.0
hearing(left)    0.0  1.0
hearing(right)   0.0  1.0


## 7. Guardado de datos procesados y transformadores

Guardamos en `data/processed/` los conjuntos listos para modelar, y en `models/` los transformadores ajustados. Esto es clave: en el notebook 06 necesitaremos **el mismo** encoder y scaler para procesar los datos de entrega exactamente igual que el entrenamiento.

In [9]:
# Datos procesados
X_train_final.to_parquet('../data/processed/X_train.parquet')
X_test_final.to_parquet('../data/processed/X_test.parquet')
X_train_scaled.to_parquet('../data/processed/X_train_scaled.parquet')
X_test_scaled.to_parquet('../data/processed/X_test_scaled.parquet')
y_train.to_frame().to_parquet('../data/processed/y_train.parquet')
y_test.to_frame().to_parquet('../data/processed/y_test.parquet')

# Transformadores (para aplicar a los datos de entrega)
joblib.dump(ohe, '../models/ohe_encoder.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump({'cat_cols': cat_cols, 'num_cols': num_cols,
             'columnas_eliminadas': columnas_a_eliminar,
             'columnas_finales': list(X_train_final.columns)},
            '../models/metadata_preprocesamiento.joblib')

print('Guardado completo ✓')
print()
print('En data/processed/: X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test')
print('En models/: ohe_encoder, scaler, metadata_preprocesamiento')

Guardado completo ✓

En data/processed/: X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test
En models/: ohe_encoder, scaler, metadata_preprocesamiento


## Conclusiones del preprocesamiento

- Eliminamos `ID` y `oral` por no aportar señal.
- **X** = 24 features, **y** = `smoking`.
- Split **80/20 estratificado** → 40.000 train / 10.000 test, manteniendo la proporción 63/37 en ambos.
- `gender` y `tartar` codificadas con OneHotEncoder (ajustado solo en train).
- Creamos dos versiones: **sin escalar** (para árboles) y **escalada** (para KNN).
- Guardamos datos procesados y transformadores para reutilizarlos en la predicción final.

**Decisión metodológica central:** todo transformador se ajusta solo con train y se aplica a test. Esto garantiza que la evaluación en test sea honesta, sin filtración de información.